In [4]:
import polars as pl
import numpy as np
import joblib
import os
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.ensemble import RandomForestClassifier
from sklearn.feature_selection import SelectKBest, f_classif
from sklearn.metrics import accuracy_score

# 1. SETUP FOLDERS
os.makedirs("models", exist_ok=True)

# 2. FAIL-SAFE DATA LOADING
def load_data(path):
    if os.path.exists(path):
        # Agar CSV mein separator ; hai to separator=";" add karo, wrna hata do
        return pl.read_csv(path) 
    else:
        print("data.CSV")
        return pl.DataFrame({
            "StudyHours": [2.5, 5.0, 8.0, 1.0, 4.0, 6.0, 3.0, 7.5, 9.0, 2.0],
            "Attendance": [70, 85, 95, 50, 75, 80, 65, 90, 98, 55],
            "Gender": ["M", "F", "M", "M", "F", "M", "F", "F", "M", "M"],
            "City": ["Delhi", "Mumbai", "Pune", "Mumbai", "Delhi", "Pune", "Mumbai", "Delhi", "Pune", "Delhi"],
            "Pass": [0, 1, 1, 0, 1, 1, 0, 1, 1, 0]
        })

df = load_data("data.csv")

# 3. CLEANING & PROCESSING
df = df.with_columns([
    pl.col("StudyHours").fill_null(pl.col("StudyHours").median()),
    (pl.col("StudyHours") + pl.col("Attendance")).alias("Engagement")
])

le = LabelEncoder()
df = df.with_columns(pl.Series("Gender", le.fit_transform(df["Gender"].to_numpy())))
df = df.to_dummies(columns=["City"])

# 4. PREPARATION
y = df["Pass"].to_numpy()
X = df.drop("Pass").to_numpy()

# 5. FEATURE SELECTION & SPLIT
X = SelectKBest(score_func=f_classif, k=3).fit_transform(X, y)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# 6. SCALING & SAVE SCALER
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)
joblib.dump(scaler, "models/scaler.pkl")

# 7. TRAINING & SAVE MODEL
model = RandomForestClassifier(random_state=42)
model.fit(X_train, y_train)
joblib.dump(model, "models/model.pkl")

# 8. VERIFICATION
preds = model.predict(X_test)
print(f"Pipeline Execution Successful! Accuracy: {accuracy_score(y_test, preds):.2f}")
print(df.head())

data.CSV
Pipeline Execution Successful! Accuracy: 1.00
shape: (5, 8)
┌────────────┬────────────┬────────┬────────────┬─────────────┬───────────┬──────┬────────────┐
│ StudyHours ┆ Attendance ┆ Gender ┆ City_Delhi ┆ City_Mumbai ┆ City_Pune ┆ Pass ┆ Engagement │
│ ---        ┆ ---        ┆ ---    ┆ ---        ┆ ---         ┆ ---       ┆ ---  ┆ ---        │
│ f64        ┆ i64        ┆ i64    ┆ u8         ┆ u8          ┆ u8        ┆ i64  ┆ f64        │
╞════════════╪════════════╪════════╪════════════╪═════════════╪═══════════╪══════╪════════════╡
│ 2.5        ┆ 70         ┆ 1      ┆ 1          ┆ 0           ┆ 0         ┆ 0    ┆ 72.5       │
│ 5.0        ┆ 85         ┆ 0      ┆ 0          ┆ 1           ┆ 0         ┆ 1    ┆ 90.0       │
│ 8.0        ┆ 95         ┆ 1      ┆ 0          ┆ 0           ┆ 1         ┆ 1    ┆ 103.0      │
│ 1.0        ┆ 50         ┆ 1      ┆ 0          ┆ 1           ┆ 0         ┆ 0    ┆ 51.0       │
│ 4.0        ┆ 75         ┆ 0      ┆ 1          ┆ 0           ┆ 0  